# Clustering: finding groups without labels

Data Scientist track, lesson DS10. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** run k-means on scaled log counts, choose the number of groups with inertia, silhouette and stability, describe each group in plain words, and say what clusters cannot prove.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import FunctionTransformer

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The people to cluster

Clustering has no labels, so the units below describe the people and score nothing: they use the whole table. The split and the day-7 model are here for Exercise 2. The four counts of the first week describe a person's behavior; the 4,531 people with no active day are a rule (all four counts are 0), not a cluster, so the clustering starts from the other people. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)       # 13,362 and 4,454 people
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)   # 10,021 and 3,341, cut from the training part
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)       # the five folds of lesson DS5


def logit():
    """The day-7 model of lesson DS2: one-hot text columns, scaled number columns, logistic regression."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


def auc_gain(p_new, p_base, y_true, resamples=2000):
    """ROC AUC of one model minus another on the same people, with a 95 percent bootstrap interval; both models are scored on the same resamples."""
    rng = np.random.default_rng(SEED)
    y_true, a, b = np.asarray(y_true), np.asarray(p_new), np.asarray(p_base)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y_true), len(y_true))
        gains.append(roc_auc_score(y_true[i], a[i]) - roc_auc_score(y_true[i], b[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(roc_auc_score(y_true, a) - roc_auc_score(y_true, b)), 3) + 0.0, round(float(low), 3) + 0.0, round(float(high), 3) + 0.0]

COUNTS = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d"]
active = table[table["active_days_7d"] > 0]                                    # the people with at least one active day in week 1
Z = StandardScaler().fit_transform(np.log1p(active[COUNTS]))                    # a log for the long tail of the counts, then every column on one scale


def stability(F, k, repeats=10):
    """The median adjusted Rand index between two k-means fits, each on a random half of the people, when both fits label all the people."""
    scores = []
    for r in range(repeats):
        order = np.random.default_rng(100 + r).permutation(len(F))
        first, second = order[:len(F) // 2], order[len(F) // 2:]
        a, b = (KMeans(k, n_init=5, random_state=SEED).fit(F[part]) for part in (first, second))
        scores.append(adjusted_rand_score(a.predict(F), b.predict(F)))
    return float(np.median(scores))


print(len(active), len(table) - len(active))

## Unit 1: Assign, move, and scale first

**Predict first.** Will k-means on the raw counts find about the same four groups as on the scaled log counts (an adjusted Rand index near 1), partly the same (about 0.5) or unrelated (near 0)?

In [ ]:
raw = KMeans(4, n_init=10, random_state=SEED).fit(active[COUNTS])               # the counts as they are
scaled = KMeans(4, n_init=10, random_state=SEED).fit(Z)                         # the counts on a log scale, each with mean 0 and standard deviation 1
print(sorted(int(n) for n in np.bincount(raw.labels_)), sorted(int(n) for n in np.bincount(scaled.labels_)))
print(round(float(adjusted_rand_score(raw.labels_, scaled.labels_)), 3))
print(pd.DataFrame(raw.cluster_centers_, columns=COUNTS).round(1).to_string(index=False))

## Unit 2: Three rulers for k

**Predict first.** On the made-up data the silhouette peaks sharply at k = 3. Will the people's silhouette peak as sharply at one value of k, or stay in a narrow band?

In [ ]:
made_up, _ = make_blobs(len(Z), n_features=4, centers=3, cluster_std=1.0, random_state=SEED)       # made-up data with three real groups, as many rows as people
ruler_rows = []
for k in range(2, 7):
    km = KMeans(k, n_init=10, random_state=SEED).fit(Z)
    km_made = KMeans(k, n_init=10, random_state=SEED).fit(made_up)
    ruler_rows.append((k, round(float(km.inertia_)), round(float(silhouette_score(Z, km.labels_, sample_size=5000, random_state=SEED)), 3),
                 round(stability(Z, k), 3), round(float(silhouette_score(made_up, km_made.labels_, sample_size=5000, random_state=SEED)), 3)))
print(pd.DataFrame(ruler_rows, columns=["k", "inertia", "silhouette", "stability", "silhouette, made-up 3 groups"]).to_string(index=False))

## Unit 3: Describe a segment in words

**Predict first.** How far apart will the churn rates of the light and heavy segments be: under 5 points, 5 to 15 points, or over 15 points?

In [ ]:
km3 = KMeans(3, n_init=10, random_state=SEED).fit(Z)
segments = active.assign(segment=km3.labels_)
order = segments.groupby("segment")["active_days_7d"].mean().sort_values().index                 # name the segments by activity, light to heavy
segments["segment"] = segments["segment"].map(dict(zip(order, ["light", "steady", "heavy"])))
profile = segments.groupby("segment").agg(people=("churned", "size"), sessions=("sessions_7d", "mean"), expenses=("expenses_7d", "mean"), active_days=("active_days_7d", "mean"),
                                           churned=("churned", "mean"), paid_social=("channel", lambda s: (s == "paid_social").mean())).loc[["light", "steady", "heavy"]].round(2)
print(profile.to_string())
quiet = table[table["active_days_7d"] == 0]
print("no active day:", len(quiet), "people, churned", round(float(quiet["churned"].mean()), 2))
print(active.groupby(active["active_days_7d"].clip(upper=3))["churned"].mean().round(2).to_dict())          # a plain rule: 1, 2 or 3 or more active days

## Unit 4: What a cluster cannot prove

**Predict first.** Will two fits on halves of a cloud with no groups at all agree (an adjusted Rand index above 0.9), agree partly or disagree?

In [ ]:
cloud = np.random.default_rng(SEED).multivariate_normal(Z.mean(axis=0), np.cov(Z.T), size=len(Z))          # one cloud with no groups by construction, with the spread and correlations of the people
stab_rows = [(k, round(stability(Z, k), 3), round(stability(cloud, k), 3)) for k in (2, 3, 4)]
print(pd.DataFrame(stab_rows, columns=["k", "stability, people", "stability, cloud with no groups"]).to_string(index=False))

## Finish it

### Exercise 1: Finish it

**Task:** **Four segments.** Group the people into four segments with k-means on the scaled log counts (`four` and `seg4` are given), and put the list [the number of people in the smallest segment, the churn rate of the segment with the highest churn rate rounded to 3 decimals] in `answer_1`.

**Expected:** `answer_1` is a list of two numbers.

In [ ]:
four = KMeans(4, n_init=10, random_state=SEED).fit(Z)          # given: k = 4 on the scaled log counts
seg4 = active.assign(segment=four.labels_)
# find the number of people in the smallest segment, and the churn rate of the segment with the highest churn rate
answer_1 = None  # replace None with [that number of people, that churn rate rounded to 3 decimals]

In [ ]:
_ = check("Exercise 1", "answer_1", [2596, 0.579], traps=[([5183, 0.346], "That is the size of the segment with the highest churn rate and the smallest churn rate. Take the smallest size and the highest rate separately.")], tol=5e-4)

<details>
<summary>Hint</summary>

`seg4["segment"].value_counts()` gives the size of every segment; `seg4.groupby("segment")["churned"].mean()` gives its churn rate.

</details>

<details>
<summary>Half-solution</summary>

```python
sizes = seg4["segment"].value_counts()
rates = seg4.groupby("segment")["churned"].mean()
answer_1 = [int(sizes.min()), round(float(rates.max()), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
sizes = seg4["segment"].value_counts()
rates = seg4.groupby("segment")["churned"].mean()
answer_1 = [int(sizes.min()), round(float(rates.max()), 3)]
```

The smallest segment has 2,596 people and the highest churn rate is 0.579. The segment with the highest churn rate is not the smallest one: it is the largest, the light people, and the four segments are cuts of the same activity scale as in unit 3.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Does a segment help predict churn?** Before you start, say in one sentence what the result looks like: three numbers close to 0, the range including it. Put the day-7 model's columns plus three distances to the centers of a k-means (k = 3, `n_init=10`, `random_state=SEED`) in one pipeline: a `ColumnTransformer` with the usual one-hot `STATIC` columns, the scaled `WEEK1` columns, and `COUNTS` going through `FunctionTransformer(np.log1p)`, `StandardScaler()` and `KMeans`, then `LogisticRegression(max_iter=1000)`. Score it and `logit()` on the five `folds` of the training part, and put [mean, lowest, highest] of the five fold-by-fold differences (segment model minus `logit()`) in `answer_2`, each rounded to 4 decimals. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of three numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.0002, -0.0015, 0.0016], traps=[([0.0006, -0.0015, 0.0019], "That puts KMeans on the raw counts. Put the log and the scaler in front of KMeans, inside the pipeline.")], tol=1e-4)

<details>
<summary>Hint</summary>

`KMeans` used as a step of a pipeline turns each person into the distances to the k centers. Give the pipeline the five `folds` through `cross_val_score(..., X_train, y_train, cv=folds, scoring="roc_auc")` for both models and subtract the two arrays.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Three numbers within two thousandths of 0, the lowest below it and the highest above it."*

```python
segment_part = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KMeans(3, n_init=10, random_state=SEED))
# ColumnTransformer([("cat", ...), ("num", ...), ("segment", segment_part, COUNTS)])
```

</details>

<details>
<summary>Solution</summary>

```python
with_segment = make_pipeline(
    ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1),
                       ("segment", make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KMeans(3, n_init=10, random_state=SEED)), COUNTS)]),
    LogisticRegression(max_iter=1000))
s_seg = cross_val_score(with_segment, X_train, y_train, cv=folds, scoring="roc_auc")
s_day7 = cross_val_score(logit(), X_train, y_train, cv=folds, scoring="roc_auc")
d_seg = s_seg - s_day7
answer_2 = [round(float(d_seg.mean()), 4) + 0.0, round(float(d_seg.min()), 4) + 0.0, round(float(d_seg.max()), 4) + 0.0]
```

The segment model gains 0.0002 over the day-7 model, with fold differences from -0.0015 to 0.0016: nothing. A segment is a function of the same four counts the model already holds, and the k-means inside each fold can only re-cut them. This is the track's usual result: the segments describe, and do not add information about who leaves.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague scales the four log counts together with the outcome `churned` and runs k-means with two groups, then reports the two groups' churn rates as proof that the segments predict churn (the code gives the gap). Change the code so that the two segments come from the four counts alone (`Z`), and put the gap between the highest and the lowest churn rate in `answer_3`, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0 and 1.

In [ ]:
with_label = StandardScaler().fit_transform(np.log1p(active[COUNTS]).assign(churned=active["churned"]))          # colleague: the four counts and the outcome
two = KMeans(2, n_init=10, random_state=SEED).fit(with_label)
rates2 = active.assign(segment=two.labels_).groupby("segment")["churned"].mean()
answer_3 = round(float(rates2.max() - rates2.min()), 3)         # colleague: "two segments whose churn rates are this far apart: the segments predict churn"
# change the code so that the segments come from the four counts alone, and give the gap between the highest and the lowest churn rate, rounded to 3 decimals

In [ ]:
_ = check("Exercise 3", "answer_3", 0.175, traps={0.269: "That gap comes from clusters that were built with the outcome as a column. Cluster the four counts alone."}, tol=5e-4)

<details>
<summary>Hint</summary>

A segment that was made from the outcome will separate the outcome, whatever the counts say. Which columns should k-means see?

</details>

<details>
<summary>Half-solution</summary>

```python
two = KMeans(2, n_init=10, random_state=SEED).fit(Z)
rates2 = active.assign(segment=two.labels_).groupby("segment")["churned"].mean()
answer_3 = round(float(rates2.max() - rates2.min()), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
two = KMeans(2, n_init=10, random_state=SEED).fit(Z)
rates2 = active.assign(segment=two.labels_).groupby("segment")["churned"].mean()
answer_3 = round(float(rates2.max() - rates2.min()), 3)
```

With the outcome as a column the two groups are 0.269 apart in churn rate, because the outcome helped to draw them. From the counts alone the gap is 0.175 (0.40 and 0.57). A segment is read against the outcome only after it is made.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")